# LeetCode 3667 — Sort Array By Absolute Value bằng Python

Link đề: https://leetcode.com/problems/sort-array-by-absolute-value/ · Độ khó: **Easy** · Chủ đề: Array, Sorting, Two Pointers, Counting Sort

> **Bài Premium:** đề viết theo hiểu biết về bài (đã đối chiếu mô tả công khai), nên đối chiếu lại nếu có tài khoản Premium.

Skeleton LeetCode:

```python
class Solution:
    def sortByAbsoluteValue(self, nums: List[int]) -> List[int]:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: sắp xếp lại `nums` theo **giá trị tuyệt đối không giảm** (giữ nguyên dấu của từng phần tử), và biết bài chấp nhận **bất kỳ** cách sắp xếp hợp lệ khi có hai phần tử cùng giá trị tuyệt đối (như `-1` và `1`).
2. Dùng `sorted(nums, key=abs)` và hiểu tham số `key`: hàm biến mỗi phần tử thành "khóa so sánh".
3. Hiểu **sắp xếp ổn định** (stable) và vì sao nó giúp kết quả **xác định** khi có khóa bằng nhau.
4. Viết 4 cách: sắp xếp chèn thủ công theo `abs`, `sorted(key=abs)`, **counting sort** theo giá trị tuyệt đối (miền `0..100`), và **heap** `heapq`; kiểm tra bằng hàm `is_valid` (không so sánh trực tiếp với một đáp án duy nhất) và test ngẫu nhiên có `random.seed`.
5. Tránh các lỗi: sắp theo **giá trị** thay vì giá trị tuyệt đối, trả về **giá trị tuyệt đối** mất dấu, `return nums.sort(...)` trả `None`, và nhầm chiều sắp xếp.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums`. Hãy **sắp xếp lại** mảng sao cho các phần tử có **giá trị tuyệt đối không giảm** từ trái sang phải. Giữ nguyên dấu của từng phần tử (`-4` vẫn là `-4`). Nếu có nhiều đáp án hợp lệ, trả về **bất kỳ** đáp án nào.

| Ví dụ | nums | Output | Giải thích |
|---|---|---|---|
| 1 | `[3,-1,-4,1,5]` | `[-1,1,3,-4,5]` | giá trị tuyệt đối `3,1,4,1,5` -> sắp `1,1,3,4,5`; `[1,-1,3,-4,5]` cũng hợp lệ |
| 2 | `[-100,100]` | `[-100,100]` | cả hai có giá trị tuyệt đối `100`; `[100,-100]` cũng hợp lệ |

Constraints: `1 <= nums.length <= 100`, `-100 <= nums[i] <= 100`.

In [1]:
nums = [3, -1, -4, 1, 5]
print([abs(x) for x in nums])
print(sorted(nums, key=abs))
print(sorted(nums))                # sắp theo giá trị: KHÁC bài này

[3, 1, 4, 1, 5]
[-1, 1, 3, -4, 5]
[-4, -1, 1, 3, 5]


## 3. Kiến thức nền

### 3.1. `abs` và giá trị tuyệt đối

`abs(x)` trả khoảng cách từ `x` đến `0`: `abs(-4) = 4`, `abs(5) = 5`, `abs(0) = 0`. Hai số đối nhau (`-1` và `1`) có cùng giá trị tuyệt đối.

In [2]:
print(abs(-4), abs(5), abs(0))
print(abs(-1) == abs(1))
print(-1 < 1, abs(-1) < abs(1))

4 5 0
True
True False


### 3.2. `sorted` và tham số `key`

`sorted(iterable, key=f)` sắp xếp các phần tử theo `f(phần tử)` và trả list **mới** (không sửa danh sách gốc). `list.sort(key=f)` sắp **tại chỗ** và trả về `None`. `key` được tính **một lần** cho mỗi phần tử.

In [3]:
nums = [3, -1, -4, 1, 5]
print(sorted(nums, key=abs), nums)         # nums không đổi
tmp = nums[:]
r = tmp.sort(key=abs)
print(r, tmp)                              # sort tại chỗ trả None

[-1, 1, 3, -4, 5] [3, -1, -4, 1, 5]
None [-1, 1, 3, -4, 5]


### 3.3. Sắp xếp ổn định (stable)

Sắp xếp **ổn định**: các phần tử có khóa bằng nhau **giữ nguyên thứ tự ban đầu**. Python `sorted` và `list.sort` đều ổn định. Với `[3,-1,-4,1,5]`, hai phần tử có `abs = 1` là `-1` (đứng trước) và `1` (đứng sau) nên kết quả ổn định là `[-1, 1, 3, -4, 5]`.

In [4]:
print(sorted([3, -1, -4, 1, 5], key=abs))
print(sorted([1, -1], key=abs), sorted([-1, 1], key=abs))      # giữ thứ tự gốc khi khóa bằng nhau

[-1, 1, 3, -4, 5]
[1, -1] [-1, 1]


### 3.4. Counting sort và heap

- **Counting sort:** khi giá trị thuộc miền nhỏ (ở đây `abs <= 100`), đếm số lần mỗi giá trị tuyệt đối xuất hiện rồi in ra theo thứ tự tăng dần: `O(n + V)`.
- **Heap** (`heapq`): `heappush`/`heappop` lấy phần tử nhỏ nhất theo khóa; thêm chỉ số vào khóa `(abs, index, value)` để tránh so sánh tiếp `value` và giữ ổn định.

In [5]:
import heapq
h = []
for i, x in enumerate([3, -1, -4, 1, 5]):
    heapq.heappush(h, (abs(x), i, x))
print([heapq.heappop(h)[2] for _ in range(5)])

cnt = [0] * 101
for x in [3, 3, 1]:
    cnt[abs(x)] += 1
print(cnt[:5])

[-1, 1, 3, -4, 5]
[0, 1, 0, 2, 0]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Làm tay với `[3,-1,-4,1,5]`:

```
giá trị tuyệt đối : 3  1  4  1  5
sắp theo |x| tăng  : 1, 1, 3, 4, 5
các phần tử tương ứng: -1, 1, 3, -4, 5      (dấu giữ nguyên)
```

Quan trọng: ta sắp **các số gốc** theo **khóa** `|x|`, không phải sắp các giá trị tuyệt đối rồi bỏ dấu. Với `|x|` bằng nhau (`-1` và `1`), thứ tự giữa chúng là tùy ý, đề chấp nhận cả hai.

## 5. Cách 1 — Brute Force / baseline

Sắp xếp **chèn** (insertion sort) thủ công theo khóa `abs`: duyệt `i = 1 .. n-1`; lấy `cur = nums[i]`; dịch các phần tử đứng trước có `abs` **lớn hơn** `abs(cur)` sang phải; chèn `cur` vào chỗ trống.

Nút thắt: `O(n^2)` so sánh trong trường hợp xấu. Với `n <= 100` vẫn tức thì, nhưng không cần thiết khi đã có `sorted`.

In [6]:
class SolutionInsertion:
    def sortByAbsoluteValue(self, nums):
        a = nums[:]
        for i in range(1, len(a)):
            cur = a[i]
            j = i - 1
            while j >= 0 and abs(a[j]) > abs(cur):       # strict > để giữ ổn định
                a[j + 1] = a[j]
                j -= 1
            a[j + 1] = cur
        return a

print(SolutionInsertion().sortByAbsoluteValue([3, -1, -4, 1, 5]))
print(SolutionInsertion().sortByAbsoluteValue([-100, 100]))

[-1, 1, 3, -4, 5]
[-100, 100]


**Độ phức tạp:** Time `O(n^2)` xấu nhất (`O(n)` nếu đã sắp), Space `O(n)` cho bản sao (hoặc `O(1)` nếu sắp tại chỗ).

## 6. Ý tưởng thuật toán / bước đột phá

**Bước đột phá:** đây là bài "sắp xếp theo khóa". Ta không cần tự cài thuật toán sắp xếp: chỉ cần cho thư viện biết khóa là `abs(x)`. `sorted(nums, key=abs)` đã sắp xếp `O(n log n)` và ổn định.

**Quy tắc "nếu… thì…":**

- Nếu `|a| < |b|` thì `a` đứng trước `b`.
- Nếu `|a| == |b|` thì **thứ tự tùy ý** (dùng ổn định để kết quả xác định).
- Dấu của phần tử không bị thay đổi.

Invariant: sau khi sắp, dãy `[abs(x) for x in result]` **không giảm** và `result` là hoán vị của `nums`.

In [7]:
def is_valid(result, nums):
    # hàm kiểm tra đáp án hợp lệ: cùng đa tập, |x| không giảm
    if sorted(result) != sorted(nums):
        return False
    return all(abs(result[i]) <= abs(result[i + 1]) for i in range(len(result) - 1))

print(is_valid([-1, 1, 3, -4, 5], [3, -1, -4, 1, 5]))
print(is_valid([1, -1, 3, -4, 5], [3, -1, -4, 1, 5]))
print(is_valid([-4, -1, 1, 3, 5], [3, -1, -4, 1, 5]))      # sắp theo giá trị: SAI

True
True
False


## 7. Các cách tiếp theo

Tổng cộng 4 cách: (1) sắp chèn thủ công; (2) `sorted(key=abs)`; (3) counting sort theo `abs`; (4) heap `heapq` với khóa `(abs, index, value)`.

### Cách 2 — `sorted(key=abs)`

Một dòng, `O(n log n)`, ổn định.

In [8]:
class SolutionSorted:
    def sortByAbsoluteValue(self, nums):
        return sorted(nums, key=abs)

print(SolutionSorted().sortByAbsoluteValue([3, -1, -4, 1, 5]))
print(SolutionSorted().sortByAbsoluteValue([-100, 100]))

[-1, 1, 3, -4, 5]
[-100, 100]


Độ phức tạp Cách 2: Time `O(n log n)`, Space `O(n)`.

### Cách 3 — Counting sort theo giá trị tuyệt đối

Vì `|x| <= 100`, dùng hai mảng `neg[0..100]` và `pos[0..100]`: `neg[a]` đếm số lần `-a`, `pos[a]` đếm số lần `a` (số `0` tính vào `pos[0]`). Rồi với `a = 0, 1, ..., 100`: thêm `neg[a]` lần `-a` rồi `pos[a]` lần `a`. Kết quả sắp tăng theo `|x|`; trong cùng `|x|` thì số âm trước (một thứ tự hợp lệ).

In [9]:
class SolutionCounting:
    def sortByAbsoluteValue(self, nums):
        neg = [0] * 101
        pos = [0] * 101
        for x in nums:
            if x < 0:
                neg[-x] += 1
            else:
                pos[x] += 1
        res = []
        for a in range(101):
            res.extend([-a] * neg[a])
            res.extend([a] * pos[a])
        return res

print(SolutionCounting().sortByAbsoluteValue([3, -1, -4, 1, 5]))
print(SolutionCounting().sortByAbsoluteValue([-100, 100]))
print(SolutionCounting().sortByAbsoluteValue([0, 0, -2, 2, 0]))

[-1, 1, 3, -4, 5]
[-100, 100]
[0, 0, 0, -2, 2]


Độ phức tạp Cách 3: Time `O(n + V)` với `V = 101`, Space `O(V + n)`. Không so sánh phần tử với nhau.

### Cách 4 — Heap `heapq`

Đẩy `(abs(x), i, x)` vào heap rồi lấy ra lần lượt; chỉ số `i` giữ ổn định và tránh so sánh `x` khi `abs` bằng nhau. (Đây là "heap sort": `O(n log n)`.)

In [10]:
import heapq

class SolutionHeap:
    def sortByAbsoluteValue(self, nums):
        h = [(abs(x), i, x) for i, x in enumerate(nums)]
        heapq.heapify(h)
        return [heapq.heappop(h)[2] for _ in range(len(h))]

print(SolutionHeap().sortByAbsoluteValue([3, -1, -4, 1, 5]))
print(SolutionHeap().sortByAbsoluteValue([-100, 100]))

[-1, 1, 3, -4, 5]
[-100, 100]


Độ phức tạp Cách 4: Time `O(n log n)` (`heapify` `O(n)` + `n` lần `heappop` `O(log n)`), Space `O(n)`.

## 8. TRACE từng bước

Trace Cách 1 (sắp chèn) với `[3,-1,-4,1,5]`:

| i | cur | mảng trước | dịch những phần tử có `abs` lớn hơn `abs(cur)` | mảng sau |
|---|---|---|---|---|
| 1 | -1 | `[3,-1,-4,1,5]` | `3` (abs 3 > 1) dịch phải | `[-1,3,-4,1,5]` |
| 2 | -4 | `[-1,3,-4,1,5]` | không (`abs(3) = 3 <= 4`) | `[-1,3,-4,1,5]` |
| 3 | 1 | `[-1,3,-4,1,5]` | `-4`, `3` dịch phải; `-1` (abs 1) không dịch (không strict >) | `[-1,1,3,-4,5]` |
| 4 | 5 | `[-1,1,3,-4,5]` | không | `[-1,1,3,-4,5]` |

In [11]:
def trace_insertion(nums):
    a = nums[:]
    print("bắt đầu:", a)
    for i in range(1, len(a)):
        cur = a[i]
        j = i - 1
        while j >= 0 and abs(a[j]) > abs(cur):
            a[j + 1] = a[j]
            j -= 1
        a[j + 1] = cur
        print(f"i={i} cur={cur} -> chèn tại {j + 1}: {a}")
    return a

trace_insertion([3, -1, -4, 1, 5])
trace_insertion([-100, 100])

bắt đầu: [3, -1, -4, 1, 5]
i=1 cur=-1 -> chèn tại 0: [-1, 3, -4, 1, 5]
i=2 cur=-4 -> chèn tại 2: [-1, 3, -4, 1, 5]
i=3 cur=1 -> chèn tại 1: [-1, 1, 3, -4, 5]
i=4 cur=5 -> chèn tại 4: [-1, 1, 3, -4, 5]
bắt đầu: [-100, 100]
i=1 cur=100 -> chèn tại 1: [-100, 100]


[-100, 100]

## 9. Code nộp LeetCode

In [12]:
from typing import List

class Solution:
    def sortByAbsoluteValue(self, nums: List[int]) -> List[int]:
        return sorted(nums, key=abs)

print(Solution().sortByAbsoluteValue([3, -1, -4, 1, 5]))
print(Solution().sortByAbsoluteValue([-100, 100]))

[-1, 1, 3, -4, 5]
[-100, 100]


## 10. Vì sao thuật toán đúng

- **Hoán vị của đầu vào:** `sorted` chỉ sắp xếp lại các phần tử có sẵn, không thêm, không bớt, không đổi dấu; kết quả chắc chắn là hoán vị của `nums`.
- **Khóa so sánh đúng:** `sorted(key=abs)` so sánh hai phần tử qua `abs`; kết quả không giảm theo khóa này, tức đúng điều kiện đề.
- **Trường hợp khóa bằng nhau:** hai phần tử cùng giá trị tuyệt đối (`-a` và `a`) được đề cho phép đứng theo **bất kỳ** thứ tự; ổn định chỉ làm kết quả xác định chứ không ảnh hưởng tính hợp lệ.
- **Cách 1:** sau bước `i`, `a[0..i]` đã sắp theo `abs` (invariant của insertion sort); khi `i = n-1` cả mảng sắp xong.
- **Cách 3 (counting sort):** duyệt `a = 0..100` theo thứ tự tăng, mỗi `a` ghi đủ số lần của `-a` và `a`; nên `|x|` không giảm và đa tập được bảo toàn (đếm đúng từng giá trị).
- **Cách 4:** `heappop` luôn lấy khóa nhỏ nhất còn lại, nên dãy lấy ra không giảm theo `abs`.

## 11. Phân tích Time / Space Complexity

`n = len(nums) <= 100`, `V = 101` (số giá trị tuyệt đối có thể).

| Cách | Time | Space |
|---|---|---|
| 1. Sắp chèn thủ công | `O(n^2)` | `O(n)` (bản sao) |
| 2. `sorted(key=abs)` | `O(n log n)` | `O(n)` |
| 3. Counting sort | `O(n + V)` | `O(n + V)` |
| 4. Heap | `O(n log n)` | `O(n)` |

Counting sort nhanh nhất khi miền giá trị nhỏ; `sorted(key=abs)` là lựa chọn mặc định khi nộp bài.

## 12. Bộ test

Vì bài chấp nhận **nhiều đáp án**, ta kiểm tra bằng hàm `is_valid` (cùng đa tập + `|x|` không giảm), **không** so sánh trực tiếp với một mảng đáp án duy nhất. Với những ca có đáp án duy nhất (không có hai số đối nhau) thì kiểm tra thêm `==`.

Test 1: ví dụ đề và edge case (`1 <= len <= 100`, `-100 <= x <= 100`).

In [13]:
impls = [SolutionInsertion(), SolutionSorted(), SolutionCounting(), SolutionHeap(), Solution()]

def in_range(nums):
    return 1 <= len(nums) <= 100 and all(-100 <= x <= 100 for x in nums)

tests = [
    ([3, -1, -4, 1, 5], None),
    ([-100, 100], None),
    ([0], [0]),
    ([-7], [-7]),
    ([5, 4, 3, 2, 1], [1, 2, 3, 4, 5]),
    ([-5, -4, -3, -2, -1], [-1, -2, -3, -4, -5]),
    ([-3, 2, -1, 0], [0, -1, 2, -3]),              # không có hai số đối nhau -> đáp án duy nhất
    ([0, 0, 0], [0, 0, 0]),
    ([2, 2, -2, -2], None),                         # nhiều phần tử cùng |x|
    ([100, -99, 98, -97], [-97, 98, -99, 100]),
    ([-100] * 100, [-100] * 100),                   # độ dài tối đa
]
for nums, exp in tests:
    assert in_range(nums)
    for im in impls:
        res = im.sortByAbsoluteValue(list(nums))
        assert is_valid(res, nums), (type(im).__name__, nums, res)
        if exp is not None:
            assert res == exp, (type(im).__name__, nums, res, exp)
print("tất cả test cố định OK:", len(tests))

tất cả test cố định OK: 11


Test 2: ngẫu nhiên có seed, đúng constraints; cả khoảng giá trị nhỏ (nhiều cặp đối nhau) lẫn khoảng đầy đủ. Ngoài `is_valid`, kiểm tra thêm: dãy `abs` của kết quả **bằng** dãy `abs` đã sắp của đầu vào (tức các cách khác nhau chỉ có thể khác nhau ở thứ tự trong nhóm cùng `|x|`).

In [14]:
import random
random.seed(3667)

for t in range(4000):
    n = random.randint(1, 100)
    hi = random.choice([1, 3, 10, 100])
    nums = [random.randint(-hi, hi) for _ in range(n)]
    assert in_range(nums)
    expected_abs = sorted(abs(x) for x in nums)
    for im in impls:
        res = im.sortByAbsoluteValue(list(nums))
        assert is_valid(res, nums), (type(im).__name__, nums)
        assert [abs(x) for x in res] == expected_abs
print("4000 test ngẫu nhiên OK")

4000 test ngẫu nhiên OK


Test 3: tính chất — không sửa đầu vào; sắp xếp **lần hai** không đổi kết quả (idempotent); hai cách ổn định (`SolutionInsertion`, `SolutionSorted`, `SolutionHeap`) cho **cùng** kết quả vì cùng quy tắc ổn định; kết quả có cùng tổng.

In [15]:
random.seed(667)
for _ in range(1500):
    n = random.randint(1, 100)
    nums = [random.randint(-20, 20) for _ in range(n)]
    snapshot = list(nums)
    res = Solution().sortByAbsoluteValue(nums)
    assert nums == snapshot
    assert Solution().sortByAbsoluteValue(res) == res
    assert sum(res) == sum(nums)
    assert SolutionInsertion().sortByAbsoluteValue(nums) == res == SolutionHeap().sortByAbsoluteValue(nums)
print("tính chất (không sửa input, idempotent, tổng, các cách ổn định trùng nhau): OK")

tính chất (không sửa input, idempotent, tổng, các cách ổn định trùng nhau): OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: sắp theo **giá trị** thay vì giá trị tuyệt đối

In [16]:
def wrong_by_value(nums):
    return sorted(nums)                         # SAI: thiếu key=abs

nums = [3, -1, -4, 1, 5]
print("sai :", wrong_by_value(nums), "hợp lệ?", is_valid(wrong_by_value(nums), nums))
print("đúng:", Solution().sortByAbsoluteValue(nums), "hợp lệ?", is_valid(Solution().sortByAbsoluteValue(nums), nums))
assert not is_valid(wrong_by_value(nums), nums)
assert is_valid(Solution().sortByAbsoluteValue(nums), nums)

sai : [-4, -1, 1, 3, 5] hợp lệ? False
đúng: [-1, 1, 3, -4, 5] hợp lệ? True


Vì sao sai: `[-4,-1,1,3,5]` có `|-4| = 4` đứng trước `|-1| = 1`, vi phạm "giá trị tuyệt đối không giảm". **Sửa:** `key=abs`.

### Lỗi 2: trả về các **giá trị tuyệt đối** (mất dấu)

In [17]:
def wrong_drop_sign(nums):
    return sorted(abs(x) for x in nums)         # SAI: đổi hết thành số không âm

nums = [3, -1, -4, 1, 5]
print("sai :", wrong_drop_sign(nums), "hợp lệ?", is_valid(wrong_drop_sign(nums), nums))
assert not is_valid(wrong_drop_sign(nums), nums)
assert wrong_drop_sign(nums) != Solution().sortByAbsoluteValue(nums)

sai : [1, 1, 3, 4, 5] hợp lệ? False


Vì sao sai: kết quả `[1,1,3,4,5]` không phải hoán vị của `nums` (mất dấu `-`). Đề yêu cầu **sắp xếp lại** các phần tử gốc. **Sửa:** `sorted(nums, key=abs)`.

### Lỗi 3: `return nums.sort(key=abs)` trả về `None`

In [18]:
def wrong_return_none(nums):
    return nums.sort(key=abs)                   # SAI: sort tại chỗ trả None

nums = [3, -1, -4, 1, 5]
r = wrong_return_none(nums[:])
print("sai :", r)
print("đúng:", Solution().sortByAbsoluteValue(nums))
assert r is None
assert Solution().sortByAbsoluteValue(nums) is not None

sai : None
đúng: [-1, 1, 3, -4, 5]


Vì sao sai: `list.sort` sửa mảng tại chỗ và **trả `None`**. **Sửa:** `nums.sort(key=abs); return nums`, hoặc `return sorted(nums, key=abs)`.

### Lỗi 4: nhầm chiều sắp xếp (giảm dần)

In [19]:
def wrong_reverse(nums):
    return sorted(nums, key=abs, reverse=True)  # SAI: đề yêu cầu không giảm

nums = [3, -1, -4, 1, 5]
print("sai :", wrong_reverse(nums), "hợp lệ?", is_valid(wrong_reverse(nums), nums))
assert not is_valid(wrong_reverse(nums), nums)

sai : [5, -4, 3, -1, 1] hợp lệ? False


Vì sao sai: `[5,-4,3,-1,1]` có `|x|` giảm dần, trái với "không giảm". **Sửa:** bỏ `reverse=True`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Sắp xếp theo một tiêu chí khác giá trị" -> `sorted(arr, key=f)`; ở đây `f = abs`.
- Bài chấp nhận **nhiều đáp án** -> viết hàm `is_valid` để test, đừng so sánh `==` với một đáp án duy nhất.
- Miền giá trị nhỏ (`|x| <= 100`) -> **counting sort** `O(n + V)`.
- `sorted` trả list mới; `list.sort` sửa tại chỗ và trả `None`.
- Sắp ổn định (stable) cho kết quả xác định khi khóa trùng.
- Cần sắp theo nhiều tiêu chí -> khóa là tuple `(abs(x), x)` (nếu muốn ép số âm trước số dương cùng `|x|`).

## 15. Biến thể của bài

1. Cùng `|x|` thì **số âm trước** (hoặc dương trước): khóa `(abs(x), x)` hoặc `(abs(x), -x)`.
2. Sắp xếp theo **bình phương** (`LeetCode 977. Squares of a Sorted Array`: trả về các bình phương, dùng hai con trỏ `O(n)` khi mảng đã sắp).
3. Sắp xếp theo **khoảng cách đến một số `t`**: `key=lambda x: abs(x - t)` (bài k closest).
4. Sắp theo số bit `1` (`1356. Sort Integers by The Number of 1 Bits`).
5. Giá trị lớn hơn nhiều: không dùng counting sort, dùng `sorted`.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Sắp chèn | dịch phần tử theo `abs` | `O(n^2)` | `O(n)` | học thuật toán sắp xếp |
| 2. `sorted(key=abs)` | thư viện + khóa | `O(n log n)` | `O(n)` | **nên nộp** |
| 3. Counting sort | đếm theo `|x|` | `O(n + V)` | `O(n + V)` | miền giá trị nhỏ |
| 4. Heap | lấy `|x|` nhỏ nhất lần lượt | `O(n log n)` | `O(n)` | khi chỉ cần lấy vài phần tử nhỏ nhất đầu tiên |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `Solution().sortByAbsoluteValue([-2, 1, -1])` có thể là gì? (viết một đáp án hợp lệ).

**Bài 2 (tính tay):** `nums = [0, -9, 8, -7]`. Sắp theo `|x|` ra gì?

**Bài 3 (điền TODO):** hoàn thành `sort_abs_neg_first(nums)` — cùng `|x|` thì số **âm đứng trước**.

**Bài 4 (tự viết lại):** viết `sorted_squares_unsorted(nums)` trả về danh sách các **bình phương** theo thứ tự tăng dần (cho mảng bất kỳ).

**Bài 5 (mini challenge):** viết `closest_to_target(nums, t)` sắp xếp `nums` theo khoảng cách đến `t` (khoảng cách bằng nhau thì số nhỏ hơn trước).

In [20]:
# Bài 1: đoán output rồi chạy để kiểm tra
print(Solution().sortByAbsoluteValue([-2, 1, -1]))

# Bài 3: TODO (KHÔNG gọi hàm này khi chạy notebook)
def sort_abs_neg_first(nums):
    # TODO: sorted(nums, key=lambda x: (abs(x), ???)) sao cho -a đứng trước a
    return None

[1, -1, -2]


## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):** lấy `|x|` của từng số, sắp theo đó nhưng giữ số gốc (có dấu).

**Mức 2 (vừa):** Bài 3: khóa `(abs(x), x)` (vì `-a < a`). Bài 4: `sorted(x * x for x in nums)`. Bài 5: khóa `(abs(x - t), x)`.

**Mức 3 (gần đáp án):** Bài 1: `|x| = 2, 1, 1`; `-2` có `|x|` lớn nhất nên đứng **cuối**, hai số còn lại theo thứ tự nào cũng được: `[1, -1, -2]` hoặc `[-1, 1, -2]`. Bài 2: `[0, -7, 8, -9]`.

## 19. Đáp án tham khảo cho bài tập

In [21]:
# Bài 1
res1 = Solution().sortByAbsoluteValue([-2, 1, -1])
assert is_valid(res1, [-2, 1, -1])
assert res1[-1] == -2

# Bài 2
assert Solution().sortByAbsoluteValue([0, -9, 8, -7]) == [0, -7, 8, -9]

# Bài 3
def sort_abs_neg_first(nums):
    return sorted(nums, key=lambda x: (abs(x), x))
assert sort_abs_neg_first([3, -1, -4, 1, 5]) == [-1, 1, 3, -4, 5]
assert sort_abs_neg_first([1, -1, 1, -1]) == [-1, -1, 1, 1]

# Bài 4
def sorted_squares_unsorted(nums):
    return sorted(x * x for x in nums)
assert sorted_squares_unsorted([3, -1, -4, 1, 5]) == [1, 1, 9, 16, 25]
assert sorted_squares_unsorted([-7]) == [49]

# Bài 5
def closest_to_target(nums, t):
    return sorted(nums, key=lambda x: (abs(x - t), x))
assert closest_to_target([1, 5, 2, 8], 4) == [5, 2, 1, 8]
assert closest_to_target([3, 5], 4) == [3, 5]
print("đáp án bài tập OK")

đáp án bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "sắp xếp theo khóa tùy chỉnh":**

```python
return sorted(nums, key=abs)            # khóa = |x|, giữ nguyên dấu, ổn định
# cần ép thứ tự phụ: key=lambda x: (abs(x), x)
```

**Checklist trước khi submit:**

- [ ] Khóa sắp xếp là `abs(x)` (không phải `x`)?
- [ ] Giữ nguyên **dấu** và trả về các phần tử gốc (không đổi thành `abs`)?
- [ ] Thứ tự **không giảm** (không dùng `reverse=True`)?
- [ ] Không dùng `return nums.sort(...)` (trả `None`)?
- [ ] Hiểu rằng với `|x|` bằng nhau, đáp án nào cũng được; test bằng `is_valid`?
- [ ] Chữ ký đúng `sortByAbsoluteValue(self, nums: List[int]) -> List[int]`?